# ROGII Wellbore Geology Prediction - Fast Pipeline
This notebook contains the complete pipeline to load data, engineer features, train a LightGBM model, and generate `submission.csv`.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
import os
import lightgbm as lgb
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
warnings.filterwarnings('ignore')

# 1. Define Data Loading (Dynamically find the dataset directory in Kaggle)
KAGGLE_DIR = Path('/kaggle/input')
for root, dirs, files in os.walk('/kaggle/input'):
    if 'train' in dirs and 'test' in dirs:
        KAGGLE_DIR = Path(root)
        break
print(f"Using dataset directory: {KAGGLE_DIR}")

def load_all_data(data_dir, data_type='train'):
    base_path = Path(data_dir) / data_type
    all_dfs = []
    horiz_files = list(base_path.glob("*__horizontal_well.csv"))
    well_ids = [f.stem.split('__')[0] for f in horiz_files]
    
    if not horiz_files:
        print(f"WARNING: No files found in {base_path}")
        
    for well_id in well_ids:
        horiz_df = pd.read_csv(base_path / f"{well_id}__horizontal_well.csv")
        type_df = pd.read_csv(base_path / f"{well_id}__typewell.csv")
        if 'TVT' in horiz_df.columns and 'TVT' in type_df.columns:
            df = pd.merge(horiz_df, type_df, on='TVT', how='left', suffixes=('_horiz', '_type'))
        else:
            df = pd.concat([horiz_df, type_df], axis=1)
        df['well_id'] = well_id
        all_dfs.append(df)
        
    if not all_dfs:
        raise ValueError(f"No data could be loaded from {base_path}. Please check the folder structure.")
        
    combined_df = pd.concat(all_dfs, ignore_index=True)
    if data_type == 'test':
        combined_df['id'] = combined_df['well_id'] + '_' + combined_df.groupby('well_id').cumcount().astype(str)
    return combined_df

In [ ]:
# 2. Feature Engineering
def add_features(df):
    df = df.copy()
    if 'Z' in df.columns:
        df['depth_diff'] = df['Z'].diff().fillna(0)
    if all(col in df.columns for col in ['X', 'Y', 'Z']):
        df['distance'] = np.sqrt(df['X'].diff()**2 + df['Y'].diff()**2 + df['Z'].diff()**2).fillna(0)
        
    sensor_cols = [col for col in ['GR_horiz'] if col in df.columns]
    for col in sensor_cols:
        df[f'{col}_diff'] = df[col].diff().fillna(0)
        df[f'{col}_rolling_mean'] = df[col].rolling(window=5, min_periods=1).mean()
        
    return df

In [ ]:
# 3. Load & Process Train Data
print("Loading training data...")
train_df = load_all_data(KAGGLE_DIR, 'train')
train_df = train_df.drop(columns=['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA'], errors='ignore')

train_featured = add_features(train_df)

numeric_cols = train_featured.select_dtypes(include=[np.number]).columns.tolist()
exclude_cols = ['TVT', 'well_id', 'TVT_input', 'Geology', 'GR_type', 'id']
feature_cols = [col for col in numeric_cols if col not in exclude_cols]

imputer = SimpleImputer(strategy='median')
train_featured[feature_cols] = imputer.fit_transform(train_featured[feature_cols])

scaler = StandardScaler()
train_featured[feature_cols] = scaler.fit_transform(train_featured[feature_cols])

X = train_featured[feature_cols]
y = train_featured['TVT']

In [ ]:
# 4. Train Model
print("Training model...")
model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
    # Remove GPU args so it works easily on standard Kaggle CPU/GPU kernels
)
model.fit(X, y)

In [ ]:
# 5. Load & Process Test Data, Predict, and Submit
print("Processing test data...")
test_df = load_all_data(KAGGLE_DIR, 'test')
test_df = test_df.drop(columns=['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA'], errors='ignore')

test_featured = add_features(test_df)

# Ensure exact feature match
for col in feature_cols:
    if col not in test_featured.columns:
        test_featured[col] = np.nan

test_featured[feature_cols] = imputer.transform(test_featured[feature_cols])
test_featured[feature_cols] = scaler.transform(test_featured[feature_cols])

X_test = test_featured[feature_cols]
predictions = model.predict(X_test)

submission = pd.DataFrame({
    'id': test_featured['id'],
    'tvt': predictions
})
submission.to_csv('submission.csv', index=False)
print("Submission saved!")
submission.head()